# Data Preprocessing

This notebook prepares the cleaned dataset for machine learning. The preprocessing steps include loading the cleaned dataset, creating a working copy, engineering features, encoding categorical variables, splitting the data into training and testing sets, and saving the processed dataset for model development.

In [45]:
# Import required libraries
import pandas as pd
import numpy as np

# Load the cleaned dataset
df = pd.read_csv("../Data/Processed_data/Kenya_Crop_Yield_Processed.csv")

# Create a working copy for preprocessing
processed_df = df.copy()

# Display dataset information
print(f"Dataset Shape: {processed_df.shape}")

# Display the first five rows
processed_df.head()

processed_df.dtypes

Dataset Shape: (757, 13)


County                       str
Crop                         str
Year                       int64
Area_ha                    int64
Production_tons            int64
Yield_t_ha               float64
NDVI                     float64
Rainfall_mm              float64
Temperature_C            float64
Elevation_m              float64
Slope_Degrees            float64
Soil_Moisture            float64
Evapotranspiration_mm    float64
dtype: object

## Identify Feature Types

Before preparing the data for machine learning, the dataset is separated into categorical and numerical features. This helps determine which preprocessing techniques should be applied to each type of variable. Categorical features will later be encoded, while numerical features may be scaled depending on the machine learning algorithm.

In [46]:
# Identify categorical and numerical columns

categorical_cols = ['County', 'Crop']

numerical_cols = [
    'Year',
    'Area_ha',
    'Production_tons',
    'NDVI',
    'Rainfall_mm',
    'Temperature_C',
    'Elevation_m',
    'Slope_Degrees',
    'Soil_Moisture',
    'Evapotranspiration_mm'
]

target = 'Yield_t_ha'

print("Categorical Features:")
print(categorical_cols)

print("\nNumerical Features:")
print(numerical_cols)

print("\nTarget Variable:")
print(target)

Categorical Features:
['County', 'Crop']

Numerical Features:
['Year', 'Area_ha', 'Production_tons', 'NDVI', 'Rainfall_mm', 'Temperature_C', 'Elevation_m', 'Slope_Degrees', 'Soil_Moisture', 'Evapotranspiration_mm']

Target Variable:
Yield_t_ha


## Identifying Features and Target Variable

The dataset is divided into categorical features, numerical features, and the target variable. This separation is important because the predictor variables will be used to train the machine learning models, while the target variable is the value the models will learn to predict.

- **Categorical features** will later be encoded into numerical format.
- **Numerical features** may be scaled depending on the machine learning algorithm.
- **Target variable** represents crop yield (t/ha).

In [47]:
# Categorical features
categorical_features = processed_df.select_dtypes(include=['string']).columns.tolist()

numerical_features = processed_df.select_dtypes(include=['number']).columns.tolist()

target = 'Yield_t_ha'
numerical_features.remove(target)

print("Categorical Features:")
print(categorical_features)

print("\nNumerical Features:")
print(numerical_features)

print("\nTarget Variable:")
print(target)

Categorical Features:
['County', 'Crop']

Numerical Features:
['Year', 'Area_ha', 'Production_tons', 'NDVI', 'Rainfall_mm', 'Temperature_C', 'Elevation_m', 'Slope_Degrees', 'Soil_Moisture', 'Evapotranspiration_mm']

Target Variable:
Yield_t_ha


## Feature and Target Separation

The dataset is separated into input features (X) and the target variable (y). The target variable is `Yield_t_ha`, while all remaining variables are treated as predictors. The `Year` column is retained temporarily because it will be used to perform a chronological train-test split before being removed from the model inputs.

In [49]:
# Separate features and target

X = processed_df.drop(columns=[
    'Yield_t_ha',
    'Area_ha',
    'Production_tons'
])

y = processed_df['Yield_t_ha']

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (557, 10)
Target shape: (557,)


## Chronological Train-Test Split

Since the objective is to predict future crop yields using historical data, the dataset is split chronologically rather than randomly. Records from earlier years (2019–2021) are used for model training, while records from later years (2022–2023) are reserved for testing. This approach better reflects a real-world forecasting scenario by evaluating the model on unseen future data.

In [50]:
# Chronological train-test split

train_data = processed_df[processed_df['Year'] <= 2021]
test_data = processed_df[processed_df['Year'] >= 2022]

# Separate features and target
X_train = train_data.drop(columns=[
    'Yield_t_ha',
    'Area_ha',
    'Production_tons'
])
y_train = train_data['Yield_t_ha']

X_test = test_data.drop(columns=[
    'Yield_t_ha',
    'Area_ha',
    'Production_tons'
])
y_test = test_data['Yield_t_ha']

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

print("\nTraining Years:", sorted(train_data['Year'].unique()))
print("Testing Years:", sorted(test_data['Year'].unique()))

Training set: (334, 10)
Testing set: (223, 10)

Training Years: [np.int64(2019), np.int64(2020), np.int64(2021)]
Testing Years: [np.int64(2022), np.int64(2023)]


## Remove the Year Feature

The `Year` column was retained only to perform a chronological train-test split. After splitting, it is removed from both the training and testing feature sets so that the models learn from environmental and agricultural variables rather than the calendar year.

In [51]:
# Remove Year from the feature sets

X_train = X_train.drop(columns=['Year'])
X_test = X_test.drop(columns=['Year'])

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining columns:")
print(X_train.columns.tolist())

Training features: (334, 9)
Testing features: (223, 9)

Training columns:
['County', 'Crop', 'NDVI', 'Rainfall_mm', 'Temperature_C', 'Elevation_m', 'Slope_Degrees', 'Soil_Moisture', 'Evapotranspiration_mm']


## Encode Categorical Variables(One-hot encoding)

Machine learning models require numerical inputs and cannot directly process categorical variables such as `County` and `Crop`. Therefore, One-Hot Encoding is applied to convert these categorical features into binary indicator variables. The encoder is fitted using the training data and then applied to the testing data to prevent data leakage.

In [52]:
from sklearn.preprocessing import OneHotEncoder

# Create the encoder
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Fit on training data and transform both datasets
encoded_train = encoder.fit_transform(X_train[['County', 'Crop']])
encoded_test = encoder.transform(X_test[['County', 'Crop']])

# Get names of the encoded columns
encoded_columns = encoder.get_feature_names_out(['County', 'Crop'])

# Convert to DataFrames
encoded_train_df = pd.DataFrame(
    encoded_train,
    columns=encoded_columns,
    index=X_train.index
)

encoded_test_df = pd.DataFrame(
    encoded_test,
    columns=encoded_columns,
    index=X_test.index
)

# Remove original categorical columns
X_train = X_train.drop(columns=['County', 'Crop'])
X_test = X_test.drop(columns=['County', 'Crop'])

# Append encoded columns
X_train = pd.concat([X_train, encoded_train_df], axis=1)
X_test = pd.concat([X_test, encoded_test_df], axis=1)

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

print(f"\nNumber of encoded features: {len(encoded_columns)}")

Training set shape: (334, 54)
Testing set shape: (223, 54)

Number of encoded features: 47


## Feature Scaling

## Feature Scaling

Machine learning algorithms often perform better when numerical features are on a similar scale. Standardization transforms each numerical feature to have a mean of approximately 0 and a standard deviation of 1.

In this project, only the numerical variables are scaled, while the one-hot encoded categorical variables are left unchanged because they already represent binary values (0 and 1).

The fitted scaler is learned from the training data and then applied to both the training and testing datasets to prevent data leakage.

In [53]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

# Numerical features to scale
numeric_features = [
    'NDVI',
    'Rainfall_mm',
    'Temperature_C',
    'Elevation_m',
    'Slope_Degrees',
    'Soil_Moisture',
    'Evapotranspiration_mm'
]

# Create the scaler
scaler = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features)
    ],
    remainder='passthrough'
)

# Fit on training data only
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Preserve feature names
scaled_feature_names = numeric_features + [
    col for col in X_train.columns if col not in numeric_features
]

# Convert back to DataFrames
X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=scaled_feature_names,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=scaled_feature_names,
    index=X_test.index
)

print("Scaled training set:", X_train_scaled.shape)
print("Scaled testing set:", X_test_scaled.shape)

print("\nFirst five scaled columns:")
print(X_train_scaled.columns[:5].tolist())
X_train_scaled.head()

Scaled training set: (334, 54)
Scaled testing set: (223, 54)

First five scaled columns:
['NDVI', 'Rainfall_mm', 'Temperature_C', 'Elevation_m', 'Slope_Degrees']


,NDVI,Rainfall_mm,Temperature_C,Elevation_m,Slope_Degrees,Soil_Moisture,Evapotranspiration_mm,County_Baringo,County_Bomet,County_Bungoma,...,County_Tharaka Nithi,County_Trans Nzoia,County_Turkana,County_Uasin Gishu,Crop_Beans,Crop_Cowpeas,Crop_Maize,Crop_Pigeon Peas,Crop_Sorghum,Crop_Wheat
0,-0.641616,-0.493248,0.696798,-0.048891,1.558228,-1.570297,-0.756979,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,1.054383,0.401093,-0.553027,0.886048,-0.133097,1.110168,1.159217,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,0.466539,0.944591,0.045966,0.240464,-0.770705,0.840681,0.304775,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3,0.485643,1.390361,0.206123,-0.355987,-1.099866,0.708833,-0.195670,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,0.589085,0.724824,0.322303,-0.156832,-0.857204,0.494486,0.240675,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0


## Save Preprocessed Data

To ensure reproducibility and avoid repeating preprocessing steps, the training and testing datasets are saved to the `Processed_data` directory. Both the original (unscaled) and scaled feature matrices are retained. The unscaled datasets will be used for tree-based models (Random Forest, Gradient Boosting, and XGBoost), while the scaled datasets will be used for Linear Regression.

In [54]:
# Save unscaled datasets
X_train.to_csv("../Data/Processed_data/X_train.csv", index=False)
X_test.to_csv("../Data/Processed_data/X_test.csv", index=False)

y_train.to_csv("../Data/Processed_data/y_train.csv", index=False)
y_test.to_csv("../Data/Processed_data/y_test.csv", index=False)

# Save scaled datasets
X_train_scaled.to_csv(
    "../Data/Processed_data/X_train_scaled.csv",
    index=False
)

X_test_scaled.to_csv(
    "../Data/Processed_data/X_test_scaled.csv",
    index=False
)

print("Preprocessed datasets saved successfully.")

Preprocessed datasets saved successfully.


In [55]:
import joblib

joblib.dump(encoder, "../models/onehot_encoder.pkl")
joblib.dump(scaler, "../models/scaler.pkl")

print("Preprocessing objects saved successfully!")

Preprocessing objects saved successfully!
